# PINN Option Pricing — Demo of All Models

This notebook demonstrates the universal wrapper API across the PINN models:

| Model | Checkpoint | What it prices |
|-------|------------|----------------|
| **2D Black–Scholes** | `trained_models/two_d.pt` | Fixed \(r\), \(\sigma\) |
| **HD Black–Scholes** | `trained_models/hd_minimal.pt` | Variable \(r\), \(\sigma\) |
| **Heston** | `trained_models/long_training_heston.pt` | Stochastic volatility |
| **1-factor Bergomi** | `trained_models/bergomi.pt` (or short in-notebook train) | Forward-variance factor \(X\) |

All loading, testing, visualization, and Greeks go through `support_tools.model_wrapper`.  
Bergomi is benchmarked with **Monte Carlo** (`Bergomi_Monte_Carlo`).

**Run from the repository root** so imports resolve correctly.

## 1. Setup

In [ ]:
from pathlib import Path

import torch
import numpy as np

from support_tools.model_wrapper import (
    load_model,
    detect_model_type,
    predict_price,
    run_slice_test,
    visualize_slice_test,
    compute_greeks,
    compare_greeks,
    get_model_label,
)
from support_tools.monte_carlo_pricing_tools import Bergomi_Monte_Carlo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load models

`load_model` auto-detects the model type and architecture from checkpoint metadata.  
If no Bergomi checkpoint exists yet, a compact network is built and trained briefly below.

In [ ]:
pinn_2d, meta_2d = load_model("trained_models/two_d.pt", device=device)
pinn_hd, meta_hd = load_model("trained_models/hd_minimal.pt", device=device)
pinn_heston, meta_heston = load_model("trained_models/long_training_heston.pt", device=device)

bergomi_path = Path("trained_models/bergomi.pt")
if bergomi_path.exists():
    pinn_bergomi, meta_bergomi = load_model(str(bergomi_path), device=device)
    print(f"Loaded Bergomi checkpoint from {bergomi_path}")
else:
    from pricing.bergomi_option_pricing import PINN as PINN_bergomi, train_network as train_bergomi

    pinn_bergomi = PINN_bergomi(
        x_min=-3.0,
        x_max=3.0,
        X_max=3.0,
        T=2.0,
        r_max=0.2,
        xi0_max=0.1,
        omega_max=2.0,
        kappa_max=5.0,
        call_put="Call",
        hidden=64,
        depth=4,
    ).to(device)
    meta_bergomi = {
        "model_type": "bergomi",
        "sigma_mode": "flat_fwd",
        "stationary": True,
    }
    print("No Bergomi checkpoint found — will train a compact model in the next cell.")

for name, pinn, meta in [
    ("2D Black–Scholes", pinn_2d, meta_2d),
    ("HD Black–Scholes", pinn_hd, meta_hd),
    ("Heston", pinn_heston, meta_heston),
    ("Bergomi (1-factor)", pinn_bergomi, meta_bergomi),
]:
    print(f"{name}")
    print(f"  type={detect_model_type(pinn).value}, call_put={pinn.call_put}")
    print(f"  hidden={pinn.hidden}, depth={pinn.depth}, T={pinn.T}")
    print(f"  checkpoint keys: {sorted(k for k in meta if k != 'model_type')}")
    print()

### 2b. Short Bergomi training (skip if checkpoint already loaded)

Increase `BERGOMI_EPOCHS` for a stronger model. A few hundred epochs is enough to exercise the API.

In [ ]:
BERGOMI_EPOCHS = 300  # raise for better accuracy vs MC

if not bergomi_path.exists():
    from pricing.bergomi_option_pricing import train_network as train_bergomi

    bergomi_path.parent.mkdir(parents=True, exist_ok=True)
    history_bergomi = train_bergomi(
        pinn_bergomi,
        N_pde=2000,
        N_boundary=1000,
        epochs=BERGOMI_EPOCHS,
        lr=1e-3,
        sigma_mode="flat_fwd",
        stationary=True,
        print_every=max(BERGOMI_EPOCHS // 10, 1),
        best_model_path=str(bergomi_path),
        save_model=True,
    )
    pinn_bergomi, meta_bergomi = load_model(str(bergomi_path), device=device)
    print(
        f"Saved {bergomi_path} | best_loss={history_bergomi['best_loss']:.3e} "
        f"@ epoch {history_bergomi['best_epoch']}"
    )
else:
    print(f"Using existing checkpoint {bergomi_path}")

## 3. Price a single option with each model

Same spot / strike / maturity where possible. Extra parameters are model-specific.

In [ ]:
S, K, tau = 100.0, 100.0, 1.0
r, sigma = 0.05, 0.2

# Shared Heston parameters for this demo
heston_params = dict(
    v=0.04,           # instantaneous variance (= sigma^2 for fair comparison)
    r=r,
    kappa=2.0,
    theta=0.04,
    sigma=0.3,        # vol-of-vol
    rho=-0.5,
    sigma_mode=meta_heston.get("sigma_mode", "mean_reverting"),
)

# Shared Bergomi parameters
bergomi_params = dict(
    X=0.0,
    r=r,
    xi0=0.04,
    omega=1.0,
    kappa=2.0,
    rho=-0.5,
    sigma_mode=meta_bergomi.get("sigma_mode", "flat_fwd"),
    stationary=meta_bergomi.get("stationary", True),
)

price_2d = predict_price(pinn_2d, S=S, K=K, tau=tau)
price_hd = predict_price(pinn_hd, S=S, K=K, tau=tau, r=r, sigma=sigma)
price_heston = predict_price(pinn_heston, S=S, K=K, tau=tau, **heston_params)
price_bergomi = predict_price(pinn_bergomi, S=S, K=K, tau=tau, **bergomi_params)

mc_bergomi, mc_se = Bergomi_Monte_Carlo(
    S0=S, K=K, T=tau, r=r, q=0.0,
    xi0=bergomi_params["xi0"],
    omega=bergomi_params["omega"],
    kappa=bergomi_params["kappa"],
    rho=bergomi_params["rho"],
    X0=bergomi_params["X"],
    n_paths=80_000,
    n_steps=100,
    stationary=bergomi_params["stationary"],
    seed=42,
    return_stderr=True,
)

print(f"ATM call  S={S}, K={K}, tau={tau}")
print(f"  2D BS   (r={pinn_2d.r}, sigma={pinn_2d.sigma}): {float(price_2d):.4f}")
print(f"  HD BS   (r={r}, sigma={sigma}):                 {float(price_hd):.4f}")
print(f"  Heston  (v={heston_params['v']}, vol-of-vol={heston_params['sigma']}): {float(price_heston):.4f}")
print(f"  Bergomi (xi0={bergomi_params['xi0']}, omega={bergomi_params['omega']}): {float(price_bergomi):.4f}")
print(f"  Bergomi MC benchmark:                          {mc_bergomi:.4f} ± {mc_se:.4f}")

## 4. Accuracy tests against ground truth

`run_slice_test` grids over \((S, \tau)\) and compares PINN prices to analytical Black–Scholes, Heston COS, or Bergomi Monte Carlo. The same function works for all models.

### 4a. 2D Black–Scholes

Uses the fixed \(r\) and \(\sigma\) stored on the model.

In [ ]:
results_2d = run_slice_test(
    pinn_2d,
    return_values=True,
    K=100,
    r=pinn_2d.r,
    sigma=pinn_2d.sigma,
    s_min=40,
    s_max=200,
    s_step=5,
    tau_min=0.1,
    tau_max=pinn_2d.T,
    n_tau=30,
)

### 4b. HD Black–Scholes

Pass any \(r\) and \(\sigma\) within the training domain (\(r \in (0, r_{\max}]\), \(\sigma \in (0, \sigma_{\max}]\)).

In [ ]:
results_hd = run_slice_test(
    pinn_hd,
    return_values=True,
    K=100,
    r=0.05,
    sigma=0.2,
    s_min=40,
    s_max=200,
    s_step=5,
    tau_min=0.1,
    tau_max=pinn_hd.T,
    n_tau=30,
)

### 4c. Heston

Uses the Fang–Oosterlee COS method as ground truth. Pass `sigma_mode` from the checkpoint so the BS baseline matches training.

In [ ]:
results_heston = run_slice_test(
    pinn_heston,
    return_values=True,
    K=100,
    v=0.04,
    r=0.05,
    kappa=2.0,
    theta=0.04,
    sigma=0.3,
    rho=-0.5,
    sigma_mode=meta_heston.get("sigma_mode", "mean_reverting"),
    s_min=50,
    s_max=200,
    s_step=10,
    tau_min=0.2,
    tau_max=3.0,
    n_tau=20,
)

### 4d. Bergomi (Monte Carlo ground truth)

Uses `Bergomi_Monte_Carlo` as the benchmark. Keep `n_mc_paths` modest for a quick demo.

In [ ]:
results_bergomi = run_slice_test(
    pinn_bergomi,
    return_values=True,
    K=100,
    X=0.0,
    r=0.05,
    xi0=0.04,
    omega=1.0,
    kappa=2.0,
    rho=-0.5,
    sigma_mode=bergomi_params["sigma_mode"],
    stationary=bergomi_params["stationary"],
    s_min=70,
    s_max=140,
    s_step=10,
    tau_min=0.25,
    tau_max=1.5,
    n_tau=6,
    n_mc_paths=15_000,
    n_mc_steps=60,
    mc_seed=42,
)

## 5. Visualize pricing errors

Interactive 3D surfaces of squared error over \((S, \tau)\). Change `values` to `"pinn"` or `"anal"` to plot prices instead of differences.

In [ ]:
visualize_slice_test(results_2d, pinn=pinn_2d, values="diff")
visualize_slice_test(results_hd, pinn=pinn_hd, values="diff")
visualize_slice_test(results_heston, pinn=pinn_heston, values="diff")
visualize_slice_test(results_bergomi, pinn=pinn_bergomi, values="diff")

## 6. Greeks (price, Delta, Theta)

Autograd-based Greeks work for all four models via `compute_greeks`. Analytic comparison against Black–Scholes is available for 2D and HD.

In [ ]:
S0, K0, tau0 = 100.0, 100.0, 1.0

# 2D — r and sigma are fixed on the model
V_2d, delta_2d, theta_2d = compute_greeks(pinn_2d, S=S0, K=K0, tau=tau0)

# HD — pass r and sigma explicitly
V_hd, delta_hd, theta_hd = compute_greeks(
    pinn_hd, S=S0, K=K0, tau=tau0, r=0.05, sigma=0.2
)

# Heston — pass full market parameters
V_h, delta_h, theta_h = compute_greeks(
    pinn_heston, S=S0, K=K0, tau=tau0, **heston_params
)

# Bergomi — pass OU factor and Bergomi parameters
V_b, delta_b, theta_b = compute_greeks(
    pinn_bergomi, S=S0, K=K0, tau=tau0, **bergomi_params
)

print(f"{'Model':<20} {'Price':>10} {'Delta':>10} {'Theta':>10}")
print("-" * 52)
for label, V, d, t in [
    ("2D Black–Scholes", V_2d, delta_2d, theta_2d),
    ("HD Black–Scholes", V_hd, delta_hd, theta_hd),
    ("Heston", V_h, delta_h, theta_h),
    ("Bergomi", V_b, delta_b, theta_b),
]:
    print(f"{label:<20} {float(V):>10.4f} {float(d):>10.4f} {float(t):>10.4f}")

### Compare 2D PINN Greeks vs analytic Black–Scholes

In [ ]:
greeks = compare_greeks(
    pinn_2d,
    K=1.0,
    r=pinn_2d.r,
    sigma=pinn_2d.sigma,
    call_put=pinn_2d.call_put,
    m_min=0.7,
    m_max=1.3,
    n_m=31,
    tau_min=0.1,
    tau_max=2.0,
    n_tau=31,
)

print(f"Max |Delta error|: {greeks['delta_diff'].abs().max().item():.4e}")
print(f"Max |Theta error|: {greeks['theta_diff'].abs().max().item():.4e}")
print(f"Mean |Delta error|: {greeks['delta_diff'].abs().mean().item():.4e}")
print(f"Mean |Theta error|: {greeks['theta_diff'].abs().mean().item():.4e}")

## 7. Side-by-side price comparison

Price the same \((S, \tau)\) grid with all models. For a fair comparison, Heston instantaneous variance and Bergomi \(\xi_0\) are set to \(\sigma^2 = 0.04\).

In [ ]:
spot_grid = np.linspace(60, 160, 21)
tau_grid = np.linspace(0.25, 2.0, 8)
K_cmp = 100.0

rows = []
for S_val in spot_grid:
    for tau_val in tau_grid:
        p2 = float(predict_price(pinn_2d, S=S_val, K=K_cmp, tau=tau_val))
        ph = float(predict_price(pinn_hd, S=S_val, K=K_cmp, tau=tau_val, r=0.05, sigma=0.2))
        phe = float(predict_price(pinn_heston, S=S_val, K=K_cmp, tau=tau_val, **heston_params))
        pb = float(predict_price(pinn_bergomi, S=S_val, K=K_cmp, tau=tau_val, **bergomi_params))
        rows.append((S_val, tau_val, p2, ph, phe, pb, abs(p2 - pb)))

arr = np.array(rows)
print(
    f"{'S':>6} {'tau':>6} {'2D':>10} {'HD':>10} {'Heston':>10} {'Bergomi':>10} {'|2D-Ber|':>10}"
)
print("-" * 72)
for S_show in [70, 100, 140]:
    for tau_show in [0.5, 1.0, 2.0]:
        mask = (np.isclose(arr[:, 0], S_show)) & (np.isclose(arr[:, 1], tau_show))
        if mask.any():
            r = arr[mask][0]
            print(
                f"{r[0]:6.0f} {r[1]:6.2f} {r[2]:10.4f} {r[3]:10.4f} "
                f"{r[4]:10.4f} {r[5]:10.4f} {r[6]:10.4f}"
            )

print()
print(f"Mean |2D − Bergomi| over grid: {arr[:, 6].mean():.4f}")

## 8. Training a new model (optional)

Each pricing module exposes `PINN` and `train_network`. Example for the 2D model (commented out — training takes a while):

```python
from pricing.two_d_option_pricing import PINN, train_network

pinn = PINN(
    x_min=-5.0, x_max=5.0, T=4.0,
    r=0.05, sigma=0.2,
    call_put="Call", hidden=50, depth=5,
).to(device)

history = train_network(
    pinn,
    epochs=5000,
    lr=7e-3,
    best_model_path="trained_models/my_two_d.pt",
)
```

Bergomi training is shown in section 2b. Analogous constructors live in
`pricing.hd_option_pricing`, `pricing.heston_option_pricing`, and
`pricing.bergomi_option_pricing`. See the README for architecture details.

## Summary

| Task | One-liner |
|------|-----------|
| Load | `pinn, meta = load_model("trained_models/….pt")` |
| Price | `predict_price(pinn, S=…, K=…, tau=…, **params)` |
| Test | `run_slice_test(pinn, return_values=True, **params)` |
| Plot | `visualize_slice_test(results, pinn=pinn)` |
| Greeks | `compute_greeks(pinn, S=…, K=…, tau=…, **params)` |
| Bergomi MC | `Bergomi_Monte_Carlo(S0, K, T, r, q, xi0, omega, kappa, rho, …)` |

The wrapper picks the correct forward signature and ground truth (BS / Heston COS / Bergomi MC) from the model type automatically.